# imports

In [ ]:
import sionna.rt as rt

import matplotlib.pyplot as plt
import mitsuba as mi
import numpy as np
import tensorflow as tf
from sim import build_rx_path, setup_drone_meshes
from sensing import (
    plot_range_doppler, ground_truth_range_velocity,
    plot_range_angle, world_to_spherical, detect_targets, score_sensing,
)
from jcas_drop import run_jcas_drop, plot_jcas_kpis

from sionna.rt import load_scene, PlanarArray, Camera

print(f"Mitsuba variant: {mi.variant()}")
print(tf.config.list_physical_devices('GPU'))

# Params

In [ ]:
preview = False
tx_power_dbm = 30       # total tx power in dBm, split across users
csi_error_std = 0.00    # stddev of simulated per-entry CSI estimation error (0 = perfect CSI)

# --- Sensing ---
sensing_on = True
plots_on = True         # graphing only; the text metrics always print

# create scene

In [ ]:
import ofdm_config

# "3.5", "10" or "28" -- rebinds carrier, bandwidth, numerology and array together
ofdm_config.select_band("28")

scene = load_scene(rt.scene.munich, merge_shapes=False)
scene.frequency = ofdm_config.CARRIER_FREQUENCY

scene.tx_array = PlanarArray(num_rows=ofdm_config.ARRAY_ROWS, num_cols=ofdm_config.ARRAY_COLS,
                             pattern="tr38901", polarization="V")
scene.rx_array = scene.tx_array


In [ ]:
cameras = {
    "cam2": Camera(position=[-56, -42, 220], look_at=[-56, -42, 0])
}

# One UAV up a Munich street canyon, x~[-71..-43], y=[-66..-20]: 13-16 m wide,
# facades 16.9-20.7 m tall, so z=15 stays inside the canyon.
uas1 = rt.Receiver(name="uas_1", position=[-63.0, -64.0, 15.0], orientation=[0.0, 0.0, 0.0])
uas1.color = [1, 1, 0]
scene.add(uas1)

num_drones = 1
num_steps = 5
dt = 1

start_points = tf.constant([
    [-63.0, -64.0, 15.0],  # street's south end, near the BS
], dtype=tf.float32)

base_velocities = tf.constant([
    [[2.4, 8.8, 0], [2.4, 8.8, 0], [2.4, 8.8, 0], [2.4, 8.8, 0], [2.4, 8.8, 0]],
], dtype=tf.float32)  # [N, num_steps, 3] -- 9.1 m/s from (-63,-64) to (-51,-20)

rx_path = build_rx_path(start_points, base_velocities, num_steps, dt, on_mismatch="error")

# One antenna serves comms and sensing alike; run_jcas_drop() reuses these below.
# Facade mount 1.5 m off the west wall, 12 m up, boresight fixed down the corridor.
bs_position = [-69.0, -64.0, 12.0]
look_at_point = [-55.0, -40.0, 15.0]


In [ ]:
# metal cube per drone: comms multipath scatterer and radar target alike
drone_meshes = setup_drone_meshes(scene, drone_radius_m=0.25)


In [ ]:
# One PathSolver solve and one OFDM frame per drone give both the RadarCube and the
# decoded comms link. See jcas_drop.py's docstring for what is simplified.
drone_names = list(scene.receivers)
tx_power_w = 10 ** (tx_power_dbm / 10) / 1000  # dBm → W

sensing_cubes = []            # [T] RadarCube per timestep
sensing_ground_truth = []     # [T] (range_m, velocity_mps) per drone
sensing_ground_truth_pos = [] # [T] drone world positions
jcas_results = []             # [T] JCASDropResult per timestep

if sensing_on:
    for t in range(num_steps):
        truths = []
        positions = []
        for i, name in enumerate(drone_names):
            pos = rx_path[i, t, :].numpy()
            # move the Receiver as well as the mesh: the comms link traces off it
            scene.receivers[name].position = pos.tolist()
            radius = float(np.array(drone_meshes[name].scaling).flat[0])
            mesh_pos = pos.copy()
            mesh_pos[2] += radius  # the z-offset setup_drone_meshes() applied at init
            drone_meshes[name].position = mesh_pos.tolist()
            # velocity is what gives PathSolver a Doppler shift to compute
            t_next = t + 1  # rx_path holds num_steps+1 points
            vel = (rx_path[i, t_next, :] - rx_path[i, t, :]).numpy() / dt
            drone_meshes[name].velocity = vel.tolist()
            truths.append(ground_truth_range_velocity(bs_position, pos, vel))
            positions.append(pos)
        sensing_ground_truth.append(truths)
        sensing_ground_truth_pos.append(positions)

        result = run_jcas_drop(
            scene, bs_position=bs_position, look_at=look_at_point, rx_names=drone_names,
            tx_power_w=tx_power_w, csi_error_std=csi_error_std,
        )
        jcas_results.append(result)
        cube = result.radar_cube
        sensing_cubes.append(cube)

        comms = result.comms[drone_names[0]]
        print(f"Sensing t={t}: peak power {cube.data.max():.3e}, "
              f"peak range {cube.range_bins[cube.data.max(axis=(0,1)).argmax()]:.1f}m  |  "
              f"comms SNR={comms.snr_db:.1f} dB, BER={comms.ber:.2e}, "
              f"throughput={comms.throughput_mbps:.1f} Mbps")


In [ ]:
# position accuracy, P_d and false alarm rate, from the CFAR detections vs. truth
if sensing_on and sensing_cubes:
    print(score_sensing(sensing_cubes, sensing_ground_truth_pos, bs_position))

In [ ]:
# SNR, BER and throughput, measured off the decoded OFDM frames
if plots_on and jcas_results:
    plot_jcas_kpis(jcas_results, scene)

In [ ]:
if plots_on and sensing_on and sensing_cubes:
    n_steps_plotted = len(sensing_cubes)
    ncols = 3
    nrows = -(-n_steps_plotted // ncols)  # ceil division

    fig = plt.figure(figsize=(6 * ncols, 5 * nrows))
    for t_idx in range(n_steps_plotted):
        ax = fig.add_subplot(nrows, ncols, t_idx + 1)
        plot_range_doppler(
            sensing_cubes[t_idx],
            max_range_m=250,
            ax=ax,
            title=f"Range-Doppler — Step {t_idx}",
            ground_truth=sensing_ground_truth[t_idx],
        )
    plt.tight_layout()
    plt.show()


In [ ]:
if plots_on and sensing_on and sensing_cubes:
    n_steps_plotted = len(sensing_cubes)
    ncols = 3
    nrows = -(-n_steps_plotted // ncols)  # ceil division

    fig = plt.figure(figsize=(6 * ncols, 5 * nrows))
    for t_idx in range(n_steps_plotted):
        ax = fig.add_subplot(nrows, ncols, t_idx + 1)
        gt_range_azimuth = [world_to_spherical(bs_position, pos)[:2] for pos in sensing_ground_truth_pos[t_idx]]
        detections = detect_targets(sensing_cubes[t_idx])
        plot_range_angle(
            sensing_cubes[t_idx],
            ground_truth=gt_range_azimuth,
            detections=detections,
            max_range_m=250,
            ax=ax,
            title=f"Range-Azimuth — Step {t_idx}",
        )
    plt.tight_layout()
    plt.show()

In [ ]:
scene.preview()